# Wagner Trait Extraction — Pydantic Deep Agent (V2)

Goal: drive `SOP.md` end-to-end with a [pydantic-deepagents](https://pydantic.dev/articles/pydantic-deep-agents) agent.

We build it up **one tiny piece at a time**: define a thing → show what it does → move on. Run the cells in order.

## 1. Install pydantic-deep

`pydantic-deep` is the deep-agents package; it pulls `pydantic-ai` with it.

In [2]:
#%pip install -q pydantic-deep -U

## 2. Confirm the OpenAI key is in the environment

pydantic-ai picks up `OPENAI_API_KEY` automatically.

In [3]:
import os
import os


assert os.getenv("OPENAI_API_KEY"), "set OPENAI_API_KEY in your shell first"
print("OPENAI_API_KEY ok")

OPENAI_API_KEY ok


## 3. Locate the project root

Everything is relative to `V2/`. The notebook lives there too.

In [4]:
from pathlib import Path
V2 = Path.cwd()
V2

PosixPath('/Users/mahdi/Documents/GitHub/ai_wagner_trait_data_extraction/V2')

## 4. Tool #1 — `read_species_inputs`

The SOP needs **all three** input files (Family / Genus / Species). The genus block carries traits like *life_form* and *leaf phyllotaxy*; the family block carries the major-group classification. Miss one and the agent misses traits.

So this tool returns all three at once, labelled, in a single call. Atomic — the agent can't forget the genus or family.

In [5]:
from pydantic_ai import RunContext
from pydantic_deep import DeepAgentDeps

async def read_species_inputs(
    ctx: RunContext[DeepAgentDeps],
    family_path: str, genus_path: str, species_path: str,
) -> dict:
    """Read the three Wagner input files for one species and return them labelled:
    {'family': ..., 'genus': ..., 'species': ...}. Paths are relative to V2/."""
    return {
        "family":  (V2 / family_path).read_text(),
        "genus":   (V2 / genus_path).read_text(),
        "species": (V2 / species_path).read_text(),
    }

### Try it — read all three files for *A. australe*

Tools take a `RunContext` as the first arg; for a manual sanity check we just pass `None`. We print the first ~150 chars of each block to confirm all three came back.

In [6]:
# # test 1
# blocks = await read_species_inputs(
#     None,
#     "test_data/hierarchy/Asteraceae/Family.md",
#     "test_data/hierarchy/Asteraceae/Acanthospermum/Genus.md",
#     "test_data/hierarchy/Asteraceae/Acanthospermum/Acanthospermum_australe.md",
# )
# for k, v in blocks.items():
#     print(f"── {k} ──\n{v[:150]}…\n")

In [7]:
# test  1
blocks = await read_species_inputs(
    None,
    "test_data/hierarchy/Asteraceae/Family.md",
    "test_data/hierarchy/Asteraceae/Ambrosia/Genus.md",
    "test_data/hierarchy/Asteraceae/Ambrosia/Ambrosia_artemisiifolia.md",
)



for k, v in blocks.items():
    print(f"── {k} ──\n{v[:150]}…\n")

    

── family ──
# 11. ASTERACEAE Sunflower family

Herbs or sometimes shrubs, rarely small to medium-sized trees or lianas, sap watery or milky. Leaves simple or comp…

── genus ──
6. *AMBROSIA* L. Ragweed

[Franseria Cav., nom. cons.]

Monoecious annual or perennial herbs or shrubs, often resinous and aromatic. Leaves simple, us…

── species ──
1. *Ambrosia artemisiifolia* L.

(nat) Common ragweed

Erect, taprooted annual herbs; stems 3–10 dm long, branched at least above, ± hirsute. Leaves p…



## 5. Tool #2 — `run_ontology`

Shell out to the V2 ontology CLI. The agent will use this for **every** `list` / `search` / `resolve` in the SOP.

In [8]:
import asyncio

async def run_ontology(ctx: RunContext[DeepAgentDeps], args: str) -> str:
    """Run the trait_ontology CLI and return its stdout (capped at 8 KB).

    `args` is the part after `python3 -m trait_ontology`. Common forms:

      list                                 -> all top-level groups
      list <group>                         -> group's traits + fields
                                              (all 15 groups are top-level)
      list <trait>                         -> a trait's allowed values + synonyms
      list <trait> <VALUE>                 -> one value's definition + synonyms
      list a,b,c                           -> BATCH form: comma-separated names,
                                              one section per name in one call
      resolve <trait> "<phrase>"           -> map a book phrase to one code
      resolve <trait> "p1, p2, p3"         -> BATCH: same trait, several phrases
      resolve-any "<phrase>"               -> map across all traits
      search "<keyword>"                   -> name/synonym/definition keyword search
      search "kw1, kw2, kw3"               -> BATCH: one section per keyword in ONE call

    PERFORMANCE — STRONGLY PREFER BATCH CALLS:
      * One call covers MANY names. Do NOT make one call per group/trait/phrase.
            list leaf_morphology,inflorescence_morphology,fruit_morphology
            resolve leaf_shape_type "ovate, lanceolate, rhombic, triangular"
      * `list <group>` returns every trait + field in that group in one call.

    OUTPUT FORMAT:
      The default human-readable tree output is ALREADY good enough for you.
      You do NOT need --json. If you do want JSON, --json must come BEFORE
      the subcommand: `--json resolve <trait> "<phrase>"`  (not after).

    Do not call `--help` or `--version` — every form you need is documented
    here.
    """
    p = await asyncio.create_subprocess_shell(
        f"python3 -m trait_ontology {args}", cwd=str(V2),
        stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT,
    )
    out, _ = await p.communicate()
    return out.decode()[:8000]


### Try it — list the trait groups

Same output the agent will see at SOP Step 1.1.

In [9]:
print(await run_ontology(None, "list"))

├── taxon_identity  (4 trait(s))
│   └── DESCRIPTION: Names of the organism (common, Hawaiian, infraspecific epithet) and its major plant group (Dicots / Monocots / Conifers / Ferns).
├── source_document  (2 trait(s))
│   └── DESCRIPTION: Provenance metadata — Wagner page number and plate references for this species. Not biological traits.
├── life_form  (1 trait(s))
│   └── DESCRIPTION: Habit (life-form) — the general appearance or manner of growth (annual herb, perennial herb, shrub, tree, vine, etc.).
├── reproductive_morphology  (3 trait(s))
│   └── DESCRIPTION: Reproductive biology — breeding system (monoecious / dioecious / polygamous / etc.) and cytology (ploidy level, chromosome number). The breeding system can be derived EITHER from a direct labelled statement in the passage ("plants dioecious", "monoecious", etc.) OR by synthesising from descriptive evidence about which flowers/florets are male vs female and where they sit on the plant. Do NOT assign breeding system based on 

## 6. Tool #3 — `save_extracted_group`

Wraps `GROUP_MODELS[name]` + `save_group(...)` so the agent never hand-writes JSON: it sends a JSON blob of field values, Pydantic validates it (rejecting bad Enum codes, wrong types, etc.), then we write the file.

In [10]:
from extraction import GROUP_MODELS, save_group

async def save_extracted_group(
    ctx: RunContext[DeepAgentDeps],
    group_name: str, family: str, genus: str, species: str,
    fields_json: str,
) -> str:
    """Build the Pydantic model for `group_name` from `fields_json` (an object of
    only the fields the passage states), validate it, and save the JSON.
    Returns the written path."""
    Model = GROUP_MODELS[group_name]
    obj = Model.model_validate_json(fields_json)
    return str(save_group(group_name, obj, family, genus, species))

### Try it — re-save the `taxon_identity` we did manually

Same JSON the manual SOP run produced; overwrites the file harmlessly.

In [11]:
demo_fields = '{"description":"DICOTS","common_name":"Spiny-bur, Paraguay bur","hawaiian_name":"kūkae-hipa, ʻihi kūkae hipa, pipili"}'
await save_extracted_group(None, "taxon_identity", "ASTERACEAE", "Acanthospermum", "australe", demo_fields)

'parsed/asteraceae/acanthospermum/australe/taxon_identity.json'

## 7. Bundle the tools the agents can call

One tool: `run_ontology`. That's it.

`read_species_inputs` is a Python helper used to assemble the user prompt below — we read the three files in plain Python and inline them into the user message. The model doesn't need a tool for that.

`save_extracted_group` is also not an agent tool — the dispatch loop calls `save_group(...)` directly after each extractor returns its typed Pydantic model. The agents never read files or write files; they reason.

In [12]:
TOOLS = [run_ontology]
[t.__name__ for t in TOOLS]


['run_ontology']

## 9. Typed output for the planner — `PlanningReport`

Instead of free-text output, the planner returns a validated Pydantic object. That gives us:
- one structured result the dispatcher can iterate (no parsing)
- the model can't return malformed plans (Pydantic rejects them)
- pre-cooked quotes for each group → per-group extractor only sees what's relevant

In [13]:
from pydantic import BaseModel, Field

class GroupPlan(BaseModel):
    """One group the planner flagged + the verbatim quote(s) that justify it
    + the planner's reason(s) for the assignment.
    Built downstream from the planner's `findings` — NOT directly produced by
    the LLM."""
    name: str = Field(description="Group name from the ontology (e.g. 'leaf_morphology')")
    quotes: list[str] = Field(description="Verbatim quote(s) from the input that justify this group")
    reasons: list[str] = Field(
        default_factory=list,
        description=(
            "Planner-supplied reason(s) for why these quotes were routed here. "
            "One entry per finding that contributed quotes. Passed through to "
            "the extractor as advisory context — the extractor may disagree."
        ),
    )


class QuoteWithSource(BaseModel):
    """A verbatim fragment from the input plus a tag for which file it came
    from. The tag lets downstream code (and humans reading the log) see how
    the planner used the taxonomic hierarchy."""
    text: str = Field(description="Verbatim fragment from the input text. Never paraphrase.")
    rank: str = Field(description="Which file this fragment came from: 'family', 'genus', or 'species'.")


class Finding(BaseModel):
    """One INFERENCE the planner makes about the passage.

    The evidence is one OR MORE verbatim quotes — they can be a single
    sentence, a short fragment, or several non-adjacent fragments combined to
    support one inference (e.g. ray florets being pistillate + disk florets
    being staminate together imply MONOECIOUS breeding_type).

    The conclusion is one OR MORE ontology groups that this evidence supports.
    """
    quotes: list[QuoteWithSource] = Field(
        description=(
            "One or more verbatim fragments from the family/genus/species text "
            "that together support this finding. Each fragment is tagged with "
            "its source rank ('family', 'genus', or 'species') so downstream "
            "code can see how the taxonomic hierarchy was used."
        ),
        min_length=1,
    )
    groups: list[str] = Field(
        description=(
            "Ontology group(s) this evidence supports. One or more group names. "
            "Multiple groups when the same evidence applies to several groups "
            "(e.g. ray florets with corolla colour and length -> both "
            "inflorescence_morphology AND outer_flower_morphology)."
        ),
        min_length=1,
    )
    reason: str = Field(
        description=(
            "One concise sentence explaining WHY this evidence supports the "
            "listed group(s). Name the specific field(s) or trait(s) it maps to. "
            "When the finding is a synthesis of multiple quotes, explain the "
            "inference (e.g. 'pistillate ray + staminate disk on the same head "
            "-> MONOECIOUS')."
        )
    )
    why_not: str = Field(
        default="",
        description=(
            "Only when an alternative group is OBVIOUSLY close at first glance. "
            "Explain why this evidence does NOT belong there. Leave EMPTY when "
            "the placement is unambiguous."
        ),
    )


class SiblingReference(BaseModel):
    """A relational claim in the passage — this species is described as
    similar to, differing from, or resembling another taxon. This is NOT
    a trait value. It is metadata about how the description was written.

    Sibling references are captured for the human reviewer and for a future
    merge-from-sibling step. They are NOT used to fill in fields downstream;
    the dispatch loop ignores them entirely.
    """
    related_taxon: str = Field(
        description=(
            "Name of the related taxon as written in the passage "
            "(e.g., 'A. conyzoides', 'Ageratum conyzoides')."
        )
    )
    quote: str = Field(
        description="The verbatim phrase from the passage that makes the relational claim."
    )
    rank: str = Field(
        description="Which file the quote came from: 'family', 'genus', or 'species'."
    )
    affected_groups: list[str] = Field(
        default_factory=list,
        description=(
            "Optional — list of ontology group names the similarity applies to, "
            "if the passage names them. Leave empty when the relational claim "
            "is unspecific (e.g. 'Very similar to X except for the key "
            "characters' — no specific groups identified)."
        ),
    )


class PlanningReport(BaseModel):
    """Planner output: a list of passage-level findings plus any sibling
    references. The LLM fills `findings` and `sibling_references`; `groups`
    is populated AFTER the run by `derive_groups_from_findings`."""
    family: str
    genus: str
    species: str
    findings: list[Finding] = Field(
        description=(
            "One entry per INFERENCE about the passage. Each entry's `quotes` "
            "field lists 1+ verbatim fragments that together support the "
            "inference. Coverage: every fact in the passage should appear in at "
            "least one finding."
        )
    )
    sibling_references: list[SiblingReference] = Field(
        default_factory=list,
        description=(
            "Relational claims that this species is similar to / differs from / "
            "resembles another taxon. NOT trait evidence — recorded for human "
            "review and for a future merge-from-sibling step. Usually empty; "
            "most species have no sibling references."
        ),
    )
    # Filled in Python after the planner returns; LLM is told not to fill it.
    groups: list[GroupPlan] = Field(
        default_factory=list,
        description="DO NOT FILL. Derived in Python from findings after the run.",
    )


def derive_groups_from_findings(findings: list[Finding]) -> list[GroupPlan]:
    """Collect findings by group name. Each GroupPlan's quotes are the verbatim
    fragments (with their `[rank]` tag) from findings whose `groups` list
    includes that name; its `reasons` are the planner's reason strings from
    those same findings (with why_not appended when the planner provided one).
    """
    by_group_quotes: dict[str, list[str]] = {}
    by_group_reasons: dict[str, list[str]] = {}
    for f in findings:
        reason_text = f.reason
        if f.why_not:
            reason_text = f"{f.reason}  (why_not: {f.why_not})"
        # Render each quote as "[rank] text" so downstream sees the source.
        rendered = [f"[{q.rank}] {q.text}" for q in f.quotes]
        for g in f.groups:
            by_group_quotes.setdefault(g, []).extend(rendered)
            by_group_reasons.setdefault(g, []).append(reason_text)
    out: list[GroupPlan] = []
    for name, quotes in by_group_quotes.items():
        seen_q = set(); unique_q: list[str] = []
        for q in quotes:
            if q not in seen_q:
                seen_q.add(q); unique_q.append(q)
        seen_r = set(); unique_r: list[str] = []
        for r in by_group_reasons.get(name, []):
            if r not in seen_r:
                seen_r.add(r); unique_r.append(r)
        out.append(GroupPlan(name=name, quotes=unique_q, reasons=unique_r))
    return out


PlanningReport.model_json_schema()["properties"].keys()


dict_keys(['family', 'genus', 'species', 'findings', 'sibling_references', 'groups'])

## 10. Construct the **planner** agent

This agent runs the SOP's *Planning phase only*. It reads the three input files, decides which trait groups apply, and returns a typed `PlanningReport` we can iterate. Extraction happens later in dedicated per-group agents (§10b onward).

`include_todo=False` — the typed return is our todo list. No need for the built-in `write_todos` tool ceremony.

In [14]:
TOOLS

[<function __main__.run_ontology(ctx: pydantic_ai._run_context.RunContext[pydantic_deep.deps.DeepAgentDeps], args: str) -> str>]

In [15]:
from pydantic_deep import create_deep_agent, StateBackend

# Pre-load the full group list ONCE so the planner doesn't waste a turn on
# `run_ontology("list")`. The list is small, stable, and identical every run.
GROUPS_LIST = await run_ontology(None, "list")

PLANNER_PROMPT = f"""\
You are the PLANNER for a Wagner botanical-trait extraction pipeline.

GOAL
Read the Family/Genus/Species text for ONE species, decide which trait
groups apply, and return a typed PlanningReport listing those groups with
the verbatim quote(s) that justify each one. Do NOT extract field values —
that's a downstream agent's job.

GROUPS AVAILABLE
The complete, authoritative list of trait groups in this ontology is below.
It is up-to-date. Use these group names exactly as written; do NOT call any
tool to fetch this list — you already have it.

{GROUPS_LIST}

INPUTS
- Family.md  -> family-level paragraph (carries major-group classification
                like DICOTS/MONOCOTS, sometimes life_form, leaf phyllotaxy)
- Genus.md   -> genus-level paragraph (often life_form, breeding_type,
                leaf phyllotaxy, inflorescence_type)
- Species.md -> species-level paragraph (the bulk of the measurements and
                most trait detail; also origin abbreviation `end`/`nat`/`PC`
                and the "in Hawai'i..." distribution sentence)

TAXONOMIC RANK — WALK TOP-DOWN, LOWER RANK OVERRIDES ON OVERLAP

Wagner's descriptions are NESTED: the family paragraph describes traits
shared by every genus, the genus paragraph describes traits shared by
every species in it, and the species paragraph describes what's specific
(or different) about THIS species. A given species inherits everything
its higher ranks say UNLESS the lower rank says otherwise.

For each flagged group, walk the hierarchy TOP-DOWN (family -> genus ->
species) and RECORD the quote(s) at each level that describe the group:

  1. Family pass — note any sentence in the family paragraph that
     describes a trait belonging to this group. Many of these will be
     too generic to keep — only record what is informative for THIS
     species. Be sparing.
  2. Genus pass — note the genus's sentence(s) for this group. These are
     usually informative: shape vocabulary, structural patterns, and
     organisation that applies to every species in the genus.
  3. Species pass — note the species's sentence(s) for this group. These
     give the species-specific values (colour, dimensions, counts).

OVERRIDE RULE (the important part)

After the walk, for each flagged group you may have quotes from multiple
ranks. KEEP ALL OF THEM in the finding's `quotes` list — do NOT drop the
genus or family quote just because the species also said something about
the group. The species rarely RESTATES what the genus said; it usually
just ADDS species-specific specifics. Both pieces of information are
needed downstream.

When the species DOES restate what a higher rank said (e.g., the
genus gave a range of colours and the species names one of them), the
species value WINS. State this in the finding's `reason` so the downstream
extractor knows — name which attribute the species overrode and which
attributes the higher rank contributed that the species did NOT restate.

Schematic example of the override pattern (NOT a real species):

  Genus paragraph says:   "<structural / shape description of the trait, plus
                           a range of possible colours or sizes>"
  Species paragraph says: "<the specific value(s) this species exhibits — often
                           just a colour or measurement, not the full structure>"

  Correct Finding:
    quotes=[<the genus sentence>, <the species sentence>]
    groups=[<the relevant group>]
    reason="Genus provides <the structural / shape information>. Species
            overrides <the specific attribute restated by the species> and
            adds <any new attribute the species contributes>. Both quotes
            are kept: genus for what the species did not restate, species
            for the species-specific values."

  WRONG — dropping the genus quote loses the structural information that the
  species did not restate:
    quotes=[<only the species sentence>]

PRACTICAL HEURISTIC

If the species sentence for a group is SHORT (a few clauses about colour,
size, count, or one or two attributes), almost certainly the corresponding
genus sentence has additional information you should ALSO include. Default
to keeping the genus sentence unless it's truly redundant.

If the family sentence is just generic family vocabulary that does not
narrow down anything specific for this species, skip it.


SIBLING REFERENCES — RELATIONAL CLAIMS ARE NOT FINDINGS

The passage may include a phrase that compares this species to another taxon
without listing the underlying traits. Examples (NOT exhaustive):
  * "Very similar to X except for the key characters"
  * "Differs from X by ..."
  * "Resembles X but with ..."
  * "As in X except ..."
  * "Closely related to X, differing in ..."

These are NOT trait findings. They are editorial shortcuts that say "look up
X for the rest of the description." Treat them as METADATA, not evidence.

Capture them in `sibling_references` (NOT in `findings`):

    sibling_references = [
      SiblingReference(
        related_taxon="<taxon name as written, e.g. 'A. conyzoides'>",
        quote="<verbatim phrase from the passage>",
        rank="species",   # or "genus" / "family"
        affected_groups=[]   # leave empty unless specific groups are named
      )
    ]

If the relational phrase ALSO lists differential characters explicitly
(e.g. "differs from X by larger leaves and pubescent stems"), those
differential characters themselves should ALSO appear as ordinary `findings`
for the relevant groups (here: leaf_morphology + stem_morphology). The
sibling reference captures the relational frame; the findings capture the
specific traits.

Most species have NO sibling references. Leave the list empty in that case.


WHEN IN DOUBT, INCLUDE THE PASSAGE
If a passage might or might not belong to a group, flag the group ANYWAY
and copy the relevant quote(s). In your analysis bullet for that quote,
use `-> TBD` instead of guessing at a field name. The downstream extractor
will see the quote and either find a field for it or mark it
`-> (no field)`. Missing data is worse than over-included data.

TOOLS
- run_ontology(args) -- see its docstring for every form. The group list
    is already in this prompt (above) — do NOT call it to fetch groups.
    Use run_ontology only when you need details INSIDE a specific group or
    trait that aren't visible in the group list. Batching reminder:
      list leaf_morphology,inflorescence_morphology,fruit_morphology
      resolve leaf_shape_type "ovate, lanceolate, rhombic"

GROUPS CAN OVERLAP — A SINGLE STRUCTURE OFTEN SPANS SEVERAL GROUPS
A single anatomical structure described in the passage may map to several
ontology groups at once. After you tentatively flag a group from a quote,
ask yourself: "are there other groups whose fields this same quote ALSO
satisfies?" Flag each one separately, repeating the same quote on each
GroupPlan.

The general rule: separate ontology groups for the same body part exist
because each captures a different ASPECT of it. If a single phrase from
the passage names a specialised structure (e.g. a floret, a leaflet) AND
also gives a property covered by a more general group (e.g. corolla
colour, whole-leaf shape), BOTH groups apply.

Worked example:
  Phrase: "ray florets ... rays yellow, ca. 1 mm long"
  This phrase names a specialised inflorescence part (the ray floret) AND
  gives corolla colour + length. Flag BOTH:
    - inflorescence_morphology  (the floret as a specialised part)
    - outer_flower_morphology             (its corolla colour + length)

Common overlap patterns (terminology in the passage triggers them):
  * specialised inflorescence parts (floret, ray, disk) + corolla detail
        -> the specialised inflorescence group AND outer_flower_morphology
  * leaves described as "lobed" or "dissected"
        -> leaf_morphology  (NOT leaflet_morphology — those are NOT leaflets)
  * compound leaves where LEAFLETS are described explicitly
        -> BOTH leaf_morphology AND leaflet_morphology
  * fruit + a persistent specialised inflorescence part (e.g. pappus on an achene)
        -> BOTH fruit_morphology AND inflorescence_morphology

ALWAYS-FLAG GROUPS (almost every species triggers these)
- taxon_identity     -- names + major-group (infer from family)
- source_document    -- page number, plate references (e.g. "Plate 7")
- life_form          -- first sentence usually has it ("Annual herbs ...")
- distribution       -- origin abbrev + "in Hawai'i..."
- reproductive_morphology -- anything like "[2n = ...]"

DO NOT FLAG groups the passage clearly doesn't mention.
For each flagged group, copy 1-3 short verbatim quotes from the input that
justify it (note which block: family / genus / species).

OUTPUT
Return a PlanningReport with these fields, IN ORDER:
  1. family, genus, species
  2. findings — a list of Finding entries. ONE FINDING PER INFERENCE you make
     about the passage. The evidence for a finding is 1+ verbatim fragments
     from the input. A fragment can be a single sentence, a short clause, or
     several NON-ADJACENT fragments combined to support one inference.

     Each Finding has:
       * quotes   — list of 1+ QuoteWithSource entries. Each entry has:
                       - text: the verbatim fragment (NEVER paraphrase)
                       - rank: which file it came from — "family", "genus",
                               or "species"
       * groups   — one OR MORE ontology group names this evidence supports
       * reason   — one concise sentence explaining WHY. Name the specific
                    field(s) or trait(s) it maps to. For multi-quote findings,
                    explain the inference.
       * why_not  — leave EMPTY unless an alternative group is OBVIOUSLY
                    close at first glance. Then say why it does NOT belong
                    there.

  The downstream `groups` list is DERIVED FROM `findings` by Python (not
  produced by you).

  Worked examples — splitting, multi-group, and synthesis from multiple
  quotes. Each quote carries its source rank (family / genus / species).

    [
      Finding(
        quotes=[
          QuoteWithSource(
            text="<verbatim sentence about inflorescence arrangement>",
            rank="species",
          ),
        ],
        groups=["inflorescence_morphology"],
        reason="<one sentence naming the trait/field>",
        why_not=""
      ),
      Finding(
        quotes=[
          QuoteWithSource(
            text="<verbatim sentence stating a head dimension>",
            rank="species",
          ),
        ],
        groups=["inflorescence_morphology"],
        reason="<one sentence naming the field>",
        why_not="<why NOT the obviously-close group, if one applies>"
      ),
      Finding(
        quotes=[
          QuoteWithSource(
            text="<verbatim sentence describing florets + their corolla>",
            rank="species",
          ),
        ],
        groups=["inflorescence_morphology", "outer_flower_morphology"],
        reason="<one sentence — same evidence applies to both groups>",
        why_not=""
      ),
      # SYNTHESIS: two non-adjacent fragments combined into ONE finding
      Finding(
        quotes=[
          QuoteWithSource(text="<first verbatim fragment>",  rank="species"),
          QuoteWithSource(text="<second verbatim fragment>", rank="species"),
        ],
        groups=["reproductive_morphology"],
        reason="<one sentence explaining the inference from the two fragments>",
        why_not=""
      ),
      # TAXONOMIC OVERRIDE: a higher-rank quote bundled with a species quote
      # because the species only partially restates what the higher rank said.
      Finding(
        quotes=[
          QuoteWithSource(text="<genus sentence with structural / shape info>", rank="genus"),
          QuoteWithSource(text="<species sentence with species-specific value>", rank="species"),
        ],
        groups=["<the relevant group>"],
        reason="Genus provides <structural information>; species overrides <restated attribute> and adds <new attribute>. Both quotes kept.",
        why_not=""
      ),
    ]

  Notes on the synthesis pattern:
    * The SAME verbatim fragment MAY appear in multiple findings (once for
      its direct group, once as part of a synthesis inference).
    * Always quote VERBATIM — never paraphrase the source text.

  COVERAGE: every fact in the passage should appear in at least one finding.
  Use the TAXONOMIC RANK rule (above) to choose family/genus/species quotes.
"""

# Tiny container that stashes the latest CostInfo emitted by on_cost_update.
class CostTracker:
    """Stash the latest CostInfo emitted by deep-agents' on_cost_update.
    The callback receives a single `CostInfo` argument; the last one fired
    has the cumulative cost for this agent's lifetime."""
    def __init__(self):
        self.latest = None
    def update(self, cost_info):
        self.latest = cost_info
    def usd(self):
        return self.latest.total_cost_usd if self.latest else None

planner_cost = CostTracker()

planner = create_deep_agent(
    model="openai-responses:gpt-5.4",
    instructions=PLANNER_PROMPT,
    tools=TOOLS,
    output_type=PlanningReport,
    on_cost_update=planner_cost.update,
    include_todo=False,
    include_subagents=False,
    include_skills=False,
    include_filesystem=False,
    include_execute=False,
    include_plan=False,
    include_memory=False,
    include_history_archive=False,
    web_search=False,
    web_fetch=False,
    thinking="medium",
)
print(f"planner prompt: {len(PLANNER_PROMPT)} chars (includes {len(GROUPS_LIST)}-char GROUPS_LIST)")
planner


planner prompt: 17082 chars (includes 4528-char GROUPS_LIST)


Agent(model=OpenAIResponsesModel(), name=None, end_strategy='early', model_settings={'anthropic_cache_instructions': True, 'anthropic_cache_tool_definitions': True, 'anthropic_cache_messages': True}, output_type=<class '__main__.PlanningReport'>)

## 10b. The user prompt

We read the three input files in plain Python (no tool call) and inline their contents into the user message. The planner sees family / genus / species text directly — one fewer round-trip and one fewer tool on the surface.

In [16]:
# # 1. Acanthospermum australe (D)
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Acanthospermum/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Acanthospermum/Acanthospermum_australe.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Acanthospermum"
# SPECIES_NAME = "australe"

# # 2. Acanthospermum hispidum (D)
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Acanthospermum/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Acanthospermum/Acanthospermum_hispidum.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Acanthospermum"
# SPECIES_NAME = "hispidum"

# # 3. Achillea millefolium (D)
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Achillea/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Achillea/Achillea_millefolium.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Achillea"
# SPECIES_NAME = "millefolium"

# # 4. Ageratina adenophora
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Ageratina/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Ageratina/Ageratina_adenophora.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Ageratina"
# SPECIES_NAME = "adenophora"

# # 5. Ageratina riparia
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Ageratina/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Ageratina/Ageratina_riparia.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Ageratina"
# SPECIES_NAME = "riparia"

# # 6. Ageratum conyzoides
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Ageratum/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Ageratum/Ageratum_conyzoides.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Ageratum"
# SPECIES_NAME = "conyzoides"

# # 7. Ageratum houstonianum
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Ageratum/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Ageratum/Ageratum_houstonianum.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Ageratum"
# SPECIES_NAME = "houstonianum"

# # 8. Ambrosia artemisiifolia
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Ambrosia/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Ambrosia/Ambrosia_artemisiifolia.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Ambrosia"
# SPECIES_NAME = "artemisiifolia"

# # 9. Anthemis cotula
# FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Asteraceae/Anthemis/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Asteraceae/Anthemis/Anthemis_cotula.md"
# FAMILY_NAME  = "ASTERACEAE"
# GENUS_NAME   = "Anthemis"
# SPECIES_NAME = "cotula"

# 10. Arctium lappa
FAMILY_PATH  = "test_data/hierarchy/Asteraceae/Family.md"
GENUS_PATH   = "test_data/hierarchy/Asteraceae/Arctium/Genus.md"
SPECIES_PATH = "test_data/hierarchy/Asteraceae/Arctium/Arctium_lappa.md"
FAMILY_NAME  = "ASTERACEAE"
GENUS_NAME   = "Arctium"
SPECIES_NAME = "lappa"

# # 11. Alsinidendron lychnoides (Caryophyllaceae)
# FAMILY_PATH  = "test_data/hierarchy/Caryophyllaceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Caryophyllaceae/Alsinidendron/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Caryophyllaceae/Alsinidendron/Alsinidendron_lychnoides.md"
# FAMILY_NAME  = "CARYOPHYLLACEAE"
# GENUS_NAME   = "Alsinidendron"
# SPECIES_NAME = "lychnoides"

# # 12. Atriplex eardleyae (Chenopodiaceae)
# FAMILY_PATH  = "test_data/hierarchy/Chenopodiaceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Chenopodiaceae/Atriplex/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Chenopodiaceae/Atriplex/Atriplex_eardleyae.md"
# FAMILY_NAME  = "CHENOPODIACEAE"
# GENUS_NAME   = "Atriplex"
# SPECIES_NAME = "eardleyae"

# # 13. Calophyllum inophyllum (Clusiaceae)  -- a tree (kamani)
# FAMILY_PATH  = "test_data/hierarchy/Clusiaceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Clusiaceae/Calophyllum/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Clusiaceae/Calophyllum/Calophyllum_inophyllum.md"
# FAMILY_NAME  = "CLUSIACEAE"
# GENUS_NAME   = "Calophyllum"
# SPECIES_NAME = "inophyllum"

# # 14. Bonamia menziesii (Convolvulaceae)  -- vine habit, morning-glory family
# FAMILY_PATH  = "test_data/hierarchy/Convolvulaceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Convolvulaceae/Bonamia/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Convolvulaceae/Bonamia/Bonamia_menziesii.md"
# FAMILY_NAME  = "CONVOLVULACEAE"
# GENUS_NAME   = "Bonamia"
# SPECIES_NAME = "menziesii"

# # 15. Alsinidendron obovatum (Caryophyllaceae)
# FAMILY_PATH  = "test_data/hierarchy/Caryophyllaceae/Family.md"
# GENUS_PATH   = "test_data/hierarchy/Caryophyllaceae/Alsinidendron/Genus.md"
# SPECIES_PATH = "test_data/hierarchy/Caryophyllaceae/Alsinidendron/Alsinidendron_obovatum.md"
# FAMILY_NAME  = "CARYOPHYLLACEAE"
# GENUS_NAME   = "Alsinidendron"
# SPECIES_NAME = "obovatum"

_blocks = await read_species_inputs(None, FAMILY_PATH, GENUS_PATH, SPECIES_PATH)

USER = (
    f"Extract every flagged trait group for *{GENUS_NAME} {SPECIES_NAME}* per the SOP.\n"
    f"Family={FAMILY_NAME}, Genus={GENUS_NAME}, Species={SPECIES_NAME}.\n\n"
    f"── family.md ──\n{_blocks['family']}\n\n"
    f"── genus.md ──\n{_blocks['genus']}\n\n"
    f"── species.md ──\n{_blocks['species']}"
)
print(USER[:800], "\n... (truncated)")
print(f"\nUSER prompt: {len(USER)} chars")


Extract every flagged trait group for *Arctium lappa* per the SOP.
Family=ASTERACEAE, Genus=Arctium, Species=lappa.

── family.md ──
# 11. ASTERACEAE Sunflower family

Herbs or sometimes shrubs, rarely small to medium-sized trees or lianas, sap watery or milky. Leaves simple or compound, alternate, opposite, or rarely whorled, stipules absent. Flowers (florets) borne in 1 to numerous dense heads with 1 to numerous sessile flowers on a common receptacle, heads subtended by an involucre of 1 to several series of bracts, sometimes clustered into secondary heads, often with a secondary involucre; receptacle flat, convex, concave to conical or cylindrical, the surface smooth, rough, or pitted, naked or chaffy (at least some florets subtended by a bract), occasionally densely bristly, the bristl 
... (truncated)

USER prompt: 7428 chars


## 11. Run the **planner** — typed plan in one shot

Single agent call. Returns a `PlanningReport` we can iterate. We stream the nodes so each tool call prints live.

In [17]:
import time
from pydantic_ai.messages import ToolCallPart, TextPart
from pydantic_ai.usage import UsageLimits

planner_log = []     # (step, seconds, in_tok, out_tok, tool_name)
planner_lines = []   # replayed into the species log in §13

def _pprint(s):
    print(s)
    planner_lines.append(s)

deps = DeepAgentDeps(backend=StateBackend())
limits = UsageLimits(request_limit=100)

t0 = time.perf_counter()
async with planner.iter(USER, deps=deps, usage_limits=limits) as run:
    step = 0
    t_prev = time.perf_counter()
    async for node in run:
        step += 1
        t_now = time.perf_counter(); dt = t_now - t_prev; t_prev = t_now
        resp = getattr(node, "model_response", None)
        in_tok = out_tok = None
        tool = ""
        if resp:
            u = getattr(resp, "usage", None)
            if u:
                in_tok  = getattr(u, "input_tokens",  None)
                out_tok = getattr(u, "output_tokens", None)
            for part in resp.parts:
                if isinstance(part, ToolCallPart):
                    tool = part.tool_name
                    args = str(part.args)[:80].replace("\n", " ")
                    if tool == "final_result":
                        _pprint(f"[{step}] {dt:5.1f}s  in={in_tok}  out={out_tok}  ✅ returning PlanningReport")
                    else:
                        _pprint(f"[{step}] {dt:5.1f}s  in={in_tok}  out={out_tok}  🔧 {tool}({args})")
                elif isinstance(part, TextPart) and part.content.strip():
                    _pprint(f"[{step}] {dt:5.1f}s  in={in_tok}  out={out_tok}  💬 {part.content[:100]}")
        else:
            _pprint(f"[{step}] {dt:5.1f}s  (tool-result)")
        planner_log.append((step, dt, in_tok, out_tok, tool))
    plan = run.result.output            # ← typed PlanningReport (findings only)
planner_secs = time.perf_counter() - t0

# Derive the per-group quote bundles from the sentence-level findings.
# (`plan.groups` is now a runtime attribute we attach here, not part of the
# typed output from the LLM.)
plan.groups = derive_groups_from_findings(plan.findings)

# Cost: pydantic-deep tracks USD via the CostTracking capability;
# planner_cost.update is called by deep-agents after each model call.
planner_usd = planner_cost.usd()

_pprint("")
_pprint(f"Planner finished in {planner_secs:.1f}s.")
if planner_usd is not None:
    _pprint(f"Planner cost: ${planner_usd:.4f}")
else:
    _pprint("Planner cost: (model pricing not in deep-agent registry)")
_pprint("")
_pprint("FINDINGS:")
for f in plan.findings:
    if len(f.quotes) == 1:
        q = f.quotes[0]
        _pprint(f"  - [{q.rank}] {q.text}")
    else:
        _pprint(f"  - (synthesis of {len(f.quotes)} quotes:)")
        for q in f.quotes:
            _pprint(f"      • [{q.rank}] {q.text}")
    _pprint(f"      groups: {', '.join(f.groups)}")
    _pprint(f"      reason: {f.reason}")
    if f.why_not:
        _pprint(f"      why_not: {f.why_not}")
_pprint("")
if plan.sibling_references:
    _pprint("")
    _pprint("SIBLING REFERENCES:")
    for sr in plan.sibling_references:
        _pprint(f"  - [{sr.rank}] \"{sr.quote}\"")
        _pprint(f"      related taxon: {sr.related_taxon}")
        if sr.affected_groups:
            _pprint(f"      affects: {', '.join(sr.affected_groups)}")
_pprint("")
_pprint(f"family={plan.family}  genus={plan.genus}  species={plan.species}")
_pprint(f"groups planned ({len(plan.groups)}):")
for g in plan.groups:
    _pprint(f"  • {g.name}  — {len(g.quotes)} quote(s)")


[1]   0.0s  (tool-result)
[2]   0.0s  (tool-result)
[3]  32.1s  in=7316  out=2072  ✅ returning PlanningReport
[4]   0.0s  (tool-result)

Planner finished in 32.1s.
Planner cost: $0.0494

FINDINGS:
  - (synthesis of 2 quotes:)
      • [family] # 11. ASTERACEAE Sunflower family
      • [species] 1. *Arctium lappa* L.  
(nat) Great burdock, gobo
      groups: taxon_identity
      reason: These quotes provide the family and species names, common names, and the origin abbreviation; the family placement supports the major plant-group identity recorded in taxon_identity.
  - [family] [page: 242]
      groups: source_document
      reason: This is source-document provenance metadata (page number), which belongs in source_document rather than a biological trait group.
  - [genus] Coarse biennial herbs.
      groups: life_form
      reason: This genus-level statement gives the inherited habit/life form for this species: a biennial herb.
  - [species] Plants 1–3 m tall.
      groups: stem_morphol

## 11b. Planner timing — where did the planner's time go?

In [18]:
import statistics as stats

llm_rows = [r for r in planner_log if r[2] is not None]
total = sum(r[1] for r in planner_log)
avg   = stats.mean(r[1] for r in llm_rows) if llm_rows else 0
first_in = llm_rows[0][2]  if llm_rows else 0
last_in  = llm_rows[-1][2] if llm_rows else 0

print(f"planner wall time : {total:6.1f}s")
print(f"planner LLM turns : {len(llm_rows)}")
print(f"avg s / LLM turn  : {avg:6.1f}s")
print(f"input tokens      : first={first_in}  last={last_in}  (grew {last_in - first_in})")

planner wall time :   32.1s
planner LLM turns : 1
avg s / LLM turn  :   32.1s
input tokens      : first=7316  last=7316  (grew 0)


## 12. Per-group **extractor factory**

One small agent per group, built fresh as we dispatch. Each one:
- has `output_type=GROUP_MODELS[name]` — Pydantic guarantees the JSON is valid
- gets the same tools as the planner, but the prompt says **use them only if needed** (the relevant ontology block is already in the user message)
- runs with a fresh, small context (~3k tokens instead of the planner's 40k)

In [19]:
from pydantic import create_model

EXTRACTOR_PROMPT = """\
You extract ONE trait group: `{group_name}`.

You will be given:
  * the quote(s) from the family/genus/species text that mention this group
  * the ontology block for `{group_name}` (its categorical traits + fields,
    with allowed values and their synonyms)
  * (optional) PLANNER NOTES — advisory text from the upstream planner
    explaining why these quotes were routed to this group. Read them, but
    treat them as a hint, NOT a directive. If the evidence in the quotes
    contradicts the planner's reason, follow the evidence.

OUTPUT — fill the ExtractedGroup fields IN ORDER:
  1. analysis  — an ENUMERATED list (one bullet per observation), each line
     in the form:
         - <verbatim phrase or feature> -> <field_name>[: <code or value>][, <field_name>...]
     Examples:
         - "rhombic-ovate to triangular" -> leaf_shape_type: [OVATE, DELTOID]
         - "1.5-3.5 cm long, 1-3 cm wide" -> leaf_dimensions: length 1.5-3.5 cm, width 1-3 cm
         - "irregularly serrate above the middle" -> leaf_margin_type: SERRATE
         - "petioles 0.3-1.5 cm long" -> petiole_length: 0.3-1.5 cm
     Cover EVERY field you intend to set in payload. If a quote mentions a
     feature that has no matching field in the ontology block, write:
         - "<phrase>" -> (no field) — <one-line reason>
     If a quote is ambiguous, say how you resolved it.
  2. payload   — a {group_name} object. EVERY field you set in payload must
     trace back to a bullet in your analysis; every bullet that names a
     field must produce a matching value in payload. The two MUST agree.
     Omit any field the passage does not state (it defaults to null).

Rules:
  * Use the controlled codes shown in the ontology block — never invent values.
  * For multiple terms in the passage ("ovate to lanceolate"), include all
    matching codes (the field's `multi: true` will accept a list).
  * Copy quantitative values verbatim (numbers + units, no conversion).
  * Most of the time you can return the object WITHOUT calling any tools —
    the ontology block already lists every allowed value + its synonyms.
  * If a phrase doesn't match anything in the ontology block, you MAY call
    run_ontology. Always BATCH related phrases into ONE call:
        resolve <trait> "phrase1, phrase2, phrase3"
    Do NOT make a separate call per phrase. Do not call `--help` / `--version`.

The shorter the path to the typed output, the better.
"""

def make_extractor(group_name: str, on_cost_update=None):
    """Build a fresh extractor agent. Its output_type is a dynamic
    ExtractedGroup(analysis: str, payload: GROUP_MODELS[group_name]) so the
    model commits its reasoning before filling the actual group fields."""
    Payload = GROUP_MODELS[group_name]
    ExtractedGroup = create_model(
        f"Extracted_{group_name}",
        analysis=(str, Field(description=(
            "Enumerated mapping from passage phrases to fields, written "
            "BEFORE the payload. Use one bullet per distinct observation. "
            "Format each bullet as:\n"
            "  - <verbatim phrase> -> <field_name>[: <code or value>][, <field_name>...]\n"
            "Cover EVERY field you intend to set in payload. If a phrase has "
            "no matching field, write `-> (no field) — <reason>`. Then derive "
            "the payload from these bullets — the payload and analysis MUST agree."
        ))),
        payload=(Payload, ...),
    )
    return create_deep_agent(
        model="openai-responses:gpt-5.4-mini",
        instructions=EXTRACTOR_PROMPT.format(group_name=group_name),
        tools=TOOLS,
        output_type=ExtractedGroup,
        on_cost_update=on_cost_update,
        include_todo=False,
        include_subagents=False,
        include_skills=False,
        include_filesystem=False,
        include_execute=False,
        include_plan=False,
        include_memory=False,
        include_history_archive=False,
        web_search=False,
        web_fetch=False,
        thinking="high",      # mirror the planner — reasoning helps catch overlaps
    )

# sanity check: build one and inspect
sample = make_extractor("taxon_identity")
sample


Agent(model=OpenAIResponsesModel(), name=None, end_strategy='early', model_settings={'anthropic_cache_instructions': True, 'anthropic_cache_tool_definitions': True, 'anthropic_cache_messages': True}, output_type=<class '__main__.Extracted_taxon_identity'>)

## 13. Dispatch — one **extractor** per planned group

For each group in the plan:
1. Pre-fetch the group's ontology with `list <group>` (one round-trip, includes subgroups).
2. Build a fresh extractor agent with `output_type = GROUP_MODELS[name]`.
3. Hand it the quotes + ontology in the user message.
4. The extractor returns a validated Pydantic model. We save it with `save_group` (no LLM cost).

Each extractor runs in a fresh small context.

We also start writing a **per-species log file** at `parsed/<family>/<genus>/<species>/<species>.log` containing the planner's tool/log lines (replayed from `planner_log`) plus everything the dispatch loop prints.

In [ ]:
import asyncio
from extraction import species_dir

# ---------------------------------------------------------------------------
# Parallel dispatch: each extractor runs as its own asyncio task.
#
# Concurrency knob — lower this if OpenAI rate-limits you, raise it on
# higher-tier accounts. 4 is a sensible default that gets ~3-4× speedup
# without hammering the API.
# ---------------------------------------------------------------------------
CONCURRENCY = 4
_semaphore = asyncio.Semaphore(CONCURRENCY)
_total_groups = len(plan.groups)
_done_count = 0
_progress_lock = asyncio.Lock()  # serialize the counter + live print


# Open the per-species log file.
sp_dir = species_dir(plan.family, plan.genus, plan.species)
sp_dir.mkdir(parents=True, exist_ok=True)
log_path = sp_dir / f"{plan.species.lower()}.log"
log_file = open(log_path, "w")

# Replay planner output to the log.
log_file.write("=" * 60 + "\n")
log_file.write(f"PLANNER — {plan.family} / {plan.genus} / {plan.species}\n")
log_file.write("=" * 60 + "\n")
for line in planner_lines:
    log_file.write(line + "\n")
log_file.write("\n" + "=" * 60 + "\n")
log_file.write(f"DISPATCH  (CONCURRENCY={CONCURRENCY})\n")
log_file.write("=" * 60 + "\n")
log_file.flush()


async def run_one_group(gp):
    """Extract one group concurrently. Returns (group_name, summary_tuple, buffered_lines).

    Each task writes ONLY to its own in-memory buffer. We drain buffers in
    plan-order after all tasks complete, so the log stays readable.
    """
    lines = []  # this task's output, drained to log_file later in plan-order
    def buf(msg=""):
        lines.append(msg)

    async with _semaphore:
        g_t0 = time.perf_counter()
        # Live status — bypasses the buffer; printed immediately so the user
        # sees activity during the parallel run.
        print(f"  ▶  {gp.name:<36} (started)", flush=True)

        # 1. Pre-fetch ontology for this group (single CLI call, no LLM).
        ontology_block = await run_ontology(None, f"list {gp.name}")

        # 2. Build extractor + prompt — its own cost tracker, isolated.
        ex_cost = CostTracker()
        extractor = make_extractor(gp.name, on_cost_update=ex_cost.update)
        reasons_block = ""
        if gp.reasons:
            reasons_block = (
                "\n\nPLANNER NOTES (advisory — you may disagree if the evidence shows otherwise):\n"
                + "\n".join(f"  - {r}" for r in gp.reasons)
            )
        user_prompt = (
            f"GROUP: {gp.name}\n\n"
            f"QUOTES FROM PASSAGE:\n" + "\n".join(f"  • {q}" for q in gp.quotes) +
            reasons_block +
            f"\n\nONTOLOGY BLOCK:\n{ontology_block}\n\n"
            f"Return an ExtractedGroup (analysis + payload) for {gp.name}."
        )

        # 3. Run extractor; collect stats + tool calls into the local buffer.
        ex_deps = DeepAgentDeps(backend=StateBackend())
        in_tok = out_tok = 0; turns = 0
        tool_calls = []
        buf(f"\n→ {gp.name}")
        async with extractor.iter(user_prompt, deps=ex_deps, usage_limits=UsageLimits(request_limit=20)) as run:
            async for node in run:
                resp = getattr(node, "model_response", None)
                if resp:
                    turns += 1
                    u = getattr(resp, "usage", None)
                    if u:
                        in_tok  += (getattr(u, "input_tokens",  0) or 0)
                        out_tok += (getattr(u, "output_tokens", 0) or 0)
                    for part in resp.parts:
                        if isinstance(part, ToolCallPart):
                            a = str(part.args)[:80].replace("\n", " ")
                            if part.tool_name == "final_result":
                                buf(f"     ✅ returning ExtractedGroup for {gp.name}")
                            else:
                                tool_calls.append((part.tool_name, a))
                                buf(f"     🔧 {part.tool_name}({a})")
            extracted = run.result.output
            analysis = extracted.analysis
            model = extracted.payload

        # 4. Save the typed model to disk (pure Python, no LLM, no contention).
        path = save_group(gp.name, model, plan.family, plan.genus, plan.species)

        dt = time.perf_counter() - g_t0
        usd = ex_cost.usd()
        cost_str = f"${usd:.4f}" if usd is not None else "$?.????"
        buf(f"     💭 {analysis}")
        buf(f"  ✓ {gp.name:<36}  {dt:5.1f}s   {turns} turn(s)  {len(tool_calls)} tool(s)  in={in_tok:>6} out={out_tok:>5}  {cost_str}")

        # Live status — counter + per-task summary, printed immediately.
        global _done_count
        async with _progress_lock:
            _done_count += 1
            print(f"  ✓  {gp.name:<36} {dt:5.1f}s  {cost_str}  ({_done_count}/{_total_groups} done)", flush=True)

        return (gp.name, (gp.name, dt, in_tok, out_tok, turns, tool_calls, usd, analysis), lines)


# ---- Run all extractors concurrently ----
print(f"\nDispatch: {_total_groups} extractor(s), concurrency={CONCURRENCY}\n")
t0 = time.perf_counter()
results = await asyncio.gather(*(run_one_group(gp) for gp in plan.groups))
dispatch_secs = time.perf_counter() - t0
print(f"\nAll extractors finished. Writing detailed per-group log to disk + below ...\n")

# ---- Drain task buffers in plan-order; build extractor_log ----
extractor_log = []
dispatch_usd_total = 0.0
for _name, summary, lines in results:
    for line in lines:
        print(line)
        log_file.write(line + "\n")
    extractor_log.append(summary)
    usd = summary[6]
    if usd is not None:
        dispatch_usd_total += usd
log_file.flush()


# ---- Totals ----
planner_usd_str = f"${planner_usd:.4f}" if planner_usd is not None else "$?.????"
total_usd_str   = f"${(planner_usd or 0) + dispatch_usd_total:.4f}"
final_lines = [
    "",
    f"All {len(plan.groups)} extractors done in {dispatch_secs:.1f}s  (concurrency={CONCURRENCY})",
    f"Costs: planner {planner_usd_str} + dispatch ${dispatch_usd_total:.4f} = TOTAL {total_usd_str}",
    f"Wall:  planner {planner_secs:.1f}s + dispatch {dispatch_secs:.1f}s = {planner_secs + dispatch_secs:.1f}s",
]
for line in final_lines:
    print(line)
    log_file.write(line + "\n")
log_file.close()
print(f"\nLog written to: {log_path}")


## 14. Final check — overall stats + on-disk files

In [21]:
total_in  = sum(r[2] for r in extractor_log) + sum((r[2] or 0) for r in planner_log)
total_out = sum(r[3] for r in extractor_log) + sum((r[3] or 0) for r in planner_log)
total_turns = sum(r[4] for r in extractor_log) + sum(1 for r in planner_log if r[2] is not None)
total_tools = sum(len(r[5]) for r in extractor_log)
total_usd   = (planner_usd or 0) + sum((r[6] or 0) for r in extractor_log)

planner_usd_disp = f"${planner_usd:.4f}" if planner_usd is not None else "$?"
print(f"planner   : {planner_secs:5.1f}s  ({planner_usd_disp})")
print(f"dispatch  : {dispatch_secs:5.1f}s  ({len(extractor_log)} extractors, ${dispatch_usd_total:.4f})")
print(f"TOTAL     : {planner_secs + dispatch_secs:5.1f}s   ${total_usd:.4f}")
print(f"LLM turns : {total_turns}")
print(f"tool calls: {total_tools}  (extractors only; planner tools printed above)")
print(f"tokens    : in={total_in}  out={total_out}")
print(f"\nslowest 5 extractors:")
for name, dt, ti, to, n, tools, usd, _analysis in sorted(extractor_log, key=lambda r: -r[1])[:5]:
    usd_str = f"${usd:.4f}" if usd is not None else "$?.????"
    print(f"  {name:<36}  {dt:5.1f}s  {n} turn(s)  {len(tools)} tool(s)  in={ti} out={to}  {usd_str}")
    for tname, targs in tools:
        print(f"      🔧 {tname}({targs})")


planner   :  32.1s  ($0.0494)
dispatch  :  51.8s  (13 extractors, $0.1651)
TOTAL     :  83.9s   $0.2144
LLM turns : 28
tool calls: 13  (extractors only; planner tools printed above)
tokens    : in=118710  out=20189

slowest 5 extractors:
  leaf_morphology                        22.4s  3 turn(s)  1 tool(s)  in=21135 out=2987  $0.0293
      🔧 run_ontology({"args":"list leaf_morphology"})
  inflorescence_morphology               21.4s  2 turn(s)  1 tool(s)  in=8935 out=2787  $0.0192
      🔧 run_ontology({"args":"search \"corymbiform\""})
  bract_involucre_morphology             20.8s  2 turn(s)  1 tool(s)  in=5185 out=2567  $0.0154
      🔧 run_ontology({"args":"list bract_involucre_morphology"})
  distribution                           20.6s  2 turn(s)  1 tool(s)  in=6839 out=2690  $0.0172
      🔧 run_ontology({"args":"list distribution"})
  outer_flower_morphology                14.4s  2 turn(s)  1 tool(s)  in=19058 out=1722  $0.0220
      🔧 run_ontology({"args":"list outer_flower_morpho

In [29]:
out = V2 / "parsed" / "asteraceae" / "acanthospermum" / "australe"
for p in sorted(out.glob("*.json")):
    print(p.name, "·", p.stat().st_size, "B")

bract_involucre_morphology.json · 232 B
distribution.json · 99 B
fruit_morphology.json · 329 B
head_and_spadix_morphology.json · 637 B
inflorescence_morphology.json · 542 B
leaf_morphology.json · 607 B
life_form.json · 47 B
outer_flower_morphology.json · 2019 B
reproductive_morphology.json · 108 B
source_document.json · 75 B
stem_morphology.json · 167 B
taxon_identity.json · 167 B
